# Synthetic Mobile Robot Fleet Dataset - overview

**Author:** Maria Karolina Silva de Souza

This notebook is an offline introduction to the locally prepared `v1.0.0` package. It loads the CSV distribution, checks table sizes and a few relationships, examines missing values, and introduces the two machine-learning tables. It does not train or benchmark a model.

> All records are synthetic. Model performance on these data is not evidence of performance in a real logistics operation.

## 1. Setup

The path logic works whether the notebook server starts in the repository root or in `examples/`. Required packages: `pandas` and `matplotlib`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

pd.set_option("display.max_columns", 60)

repository_root = Path("..") if (Path("..") / "data").exists() else Path(".")
data_root = (repository_root / "data").resolve()
assert data_root.exists(), f"Data directory not found: {data_root}"
data_root

## 2. Load the eight relational/event tables

In [ ]:
raw_names = [
    "MODELOS_ROBO",
    "ROBOS",
    "AMBIENTES",
    "ROTAS",
    "CONDICOES_AMBIENTAIS",
    "HISTORICO_ROBOS",
    "MISSOES",
    "MANUTENCOES",
]

tables = {
    name: pd.read_csv(data_root / "raw" / f"{name}.csv")
    for name in raw_names
}

table_summary = pd.DataFrame(
    [{"table": name, "rows": len(frame), "columns": frame.shape[1]} for name, frame in tables.items()]
).set_index("table")
table_summary.loc["TOTAL_BASE_ROWS", "rows"] = table_summary["rows"].sum()
table_summary

The eight base tables contain 46,378 data rows. The analytical tables are derived views and are counted separately.

## 3. Inspect a validated relational join

Each mission references one robot and one route. `validate="many_to_one"` makes pandas raise an error if the parent key is not unique.

In [ ]:
missions = tables["MISSOES"].copy()
robots = tables["ROBOS"]
routes = tables["ROTAS"]
models = tables["MODELOS_ROBO"]

mission_context = (
    missions
    .merge(robots, on="robo_id", how="left", validate="many_to_one")
    .merge(models, on="modelo_id", how="left", validate="many_to_one")
    .merge(routes, on="rota_id", how="left", validate="many_to_one")
)

assert len(mission_context) == len(missions)
assert mission_context[["modelo_id", "setor_origem_id", "setor_destino_id"]].notna().all().all()

mission_context[[
    "missao_id", "robo_id", "nome_modelo", "rota_id",
    "setor_origem_id", "setor_destino_id", "carga_kg", "status_missao"
]].head()

Important nuance: `condicao_inicio_id` points to the latest available environmental snapshot for the **route destination** at mission start. It is not a snapshot of the origin sector.

## 4. Missing values

Some sensor values are deliberately missing for data-quality and imputation exercises. Other blanks are structural: for example, collision time is blank when there was no collision, and next-maintenance targets can be blank near the end of the observation window.

In [ ]:
missing_missions = missions.isna().sum().loc[lambda values: values.gt(0)].sort_values(ascending=False)
missing_missions.to_frame("missing_rows")

In [ ]:
sensor_missing = missions[[
    "qualidade_sinal_wifi_durante_pct",
    "temperatura_robo_durante_c",
]].isna().sum()

ax = sensor_missing.plot(kind="bar", color=["#1b6c80", "#d97706"], figsize=(8, 4))
ax.set_title("Intentional missing mission-sensor values")
ax.set_xlabel("")
ax.set_ylabel("Missing rows")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

## 5. Basic temporal overview

In [ ]:
missions["data_hora_inicio"] = pd.to_datetime(missions["data_hora_inicio"], errors="raise")
monthly_missions = missions.set_index("data_hora_inicio").resample("MS").size()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
monthly_missions.plot(ax=axes[0], color="#1b6c80", marker="o")
axes[0].set_title("Missions by month")
axes[0].set_xlabel("Month")
axes[0].set_ylabel("Missions")

missions["status_missao"].value_counts().plot(kind="bar", ax=axes[1], color="#d97706")
axes[1].set_title("Mission outcome status")
axes[1].set_xlabel("")
axes[1].set_ylabel("Missions")
axes[1].tick_params(axis="x", rotation=25)
plt.tight_layout()
plt.show()

## 6. Analysis-ready mission table

`BASE_ML_MISSOES` has one row per mission. Columns ending in `_alvo` are outcomes and must be excluded from the feature matrix for pre-mission prediction.

In [ ]:
ml_missions = pd.read_csv(data_root / "analytical" / "BASE_ML_MISSOES.csv")
target_columns = [column for column in ml_missions.columns if column.endswith("_alvo")]
identifier_columns = ["missao_id"]
predictor_columns = [
    column for column in ml_missions.columns
    if column not in target_columns + identifier_columns
]

assert len(ml_missions) == len(missions) == 10_000
pd.Series({
    "rows": len(ml_missions),
    "candidate_predictor_columns": len(predictor_columns),
    "target_columns": len(target_columns),
})

In [ ]:
ml_missions[target_columns].describe(include="all").T

Use `data/dictionary.csv` to refine eligibility by `momento_disponivel`, `pode_usar_pre_missao`, `pode_usar_online`, and `pode_usar_pos_missao`. A suffix rule is a useful first guard, not a substitute for reviewing the dictionary.

## 7. Analysis-ready maintenance table

`BASE_ML_MANUTENCAO` has one row per robot per Monday: 78 reference weeks times 60 robots. Future labels can be blank near the end of the series when the next maintenance event lies outside the observed window.

In [ ]:
ml_maintenance = pd.read_csv(
    data_root / "analytical" / "BASE_ML_MANUTENCAO.csv",
    parse_dates=["data_referencia"],
)

coverage = pd.Series({
    "rows": len(ml_maintenance),
    "robots": ml_maintenance["robo_id"].nunique(),
    "reference_weeks": ml_maintenance["data_referencia"].nunique(),
    "positive_7_day_failure_labels": int(ml_maintenance["falha_nos_proximos_7_dias_alvo"].sum()),
    "missing_next_maintenance_targets": int(ml_maintenance["custo_proxima_manutencao_brl_alvo"].isna().sum()),
})
assert coverage["rows"] == 4_680
assert coverage["robots"] == 60
assert coverage["reference_weeks"] == 78
coverage

## 8. Next steps

1. Read `docs/DATASET_CARD.md` and `docs/LIMITATIONS.md`.
2. Select predictors from the data dictionary using the intended prediction time.
3. Split temporal problems chronologically rather than randomly when appropriate.
4. Report class balance and the treatment of missing values.
5. Treat results as synthetic-data experiments, not empirical claims about real robots or companies.

The dataset and documentation are covered by `LICENSE-DATA.md`. The executable code in this notebook is not granted a software license by that data license.